In [0]:
dbutils.widgets.text("Source_system","")
dbutils.widgets.text("Table_name","")
dbutils.widgets.text("Landing_timestamp","")
dbutils.widgets.text("Source_file_format","")

source_system = dbutils.widgets.get("Source_system")
table_name = dbutils.widgets.get("Table_name")
landing_timestamp = dbutils.widgets.get("Landing_timestamp")
source_file_format = dbutils.widgets.get("Source_file_format")

import os

landing_path = f"/Volumes/caresyncadb/landing-2/landing2-volume/{source_system}/{table_name}/{landing_timestamp}/"

# Exit gracefully (without failure) if the path or any parent directory does not exist, or it contains no files
if not os.path.isdir(landing_path) or len(os.listdir(landing_path)) == 0:
    print(f"No files were found at {landing_path}")
    dbutils.notebook.exit("No files were found")

In [0]:
if source_file_format == "csv":
  df = spark.read.format("csv").option("header","true").option("inferSchema","true").load(landing_path)
else:
  df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql import functions as f
updated_df = df.withColumn("Landing_timestamp",f.lit(landing_timestamp))\
    .withColumn("current_timestamp",f.current_timestamp())

hospital_id,hospital_name,city,state,hospital_type,bed_count,created_at,updated_at,Landing_timestamp,current_timestamp
1,Austin General Hospital,Austin,TX,General,155,2026-10-11T14:02:49.834Z,2026-10-11T14:02:49.834Z,2026-10-12_05:53:47,2026-10-14T11:49:33.910Z
2,Dallas Medical Center,Dallas,TX,Specialty,153,2026-10-11T14:02:49.834Z,2026-10-11T14:02:49.834Z,2026-10-12_05:53:47,2026-10-14T11:49:33.910Z
3,Houston Regional Hospital,Houston,TX,Teaching,353,2026-10-11T14:02:49.834Z,2026-10-11T14:02:49.834Z,2026-10-12_05:53:47,2026-10-14T11:49:33.910Z
4,San Antonio Community Hospital,San Antonio,TX,Community,452,2026-10-11T14:02:49.834Z,2026-10-11T14:02:49.834Z,2026-10-12_05:53:47,2026-10-14T11:49:33.910Z
5,Fort Worth Health Center,Fort Worth,TX,Regional,166,2026-10-11T14:02:49.834Z,2026-10-11T14:02:49.834Z,2026-10-12_05:53:47,2026-10-14T11:49:33.910Z
6,Phoenix Care Center,Phoenix,AZ,General,154,2026-10-11T14:02:49.834Z,2026-10-11T14:02:49.834Z,2026-10-12_05:53:47,2026-10-14T11:49:33.910Z
7,Denver Memorial Hospital,Denver,CO,Specialty,200,2026-10-11T14:02:49.834Z,2026-10-11T14:02:49.834Z,2026-10-12_05:53:47,2026-10-14T11:49:33.910Z
8,Seattle General Hospital,Seattle,WA,Teaching,157,2026-10-11T14:02:49.834Z,2026-10-11T14:02:49.834Z,2026-10-12_05:53:47,2026-10-14T11:49:33.910Z
9,Portland Medical Center,Portland,OR,Community,177,2026-10-11T14:02:49.834Z,2026-10-11T14:02:49.834Z,2026-10-12_05:53:47,2026-10-14T11:49:33.910Z
10,Chicago Regional Hospital,Chicago,IL,Regional,309,2026-10-11T14:02:49.834Z,2026-10-11T14:02:49.834Z,2026-10-12_05:53:47,2026-10-14T11:49:33.910Z


In [0]:
updated_df.write.mode("append").saveAsTable(f"caresyncadb.bronze.{table_name}")